# Teste do pipeline Datajud — ano 2000

Notebook isolado para validar a conexão com a API, a transformação dos registros e a gravação em Parquet.

Por segurança, o teste usa 1.000 registros por padrão e grava em caminhos separados:

- bruto: `data/raw/datajud_teste/2000/`;
- Parquet: `data/processed/datajud/teste/tabela_principal_helios_2000_teste.parquet`.

O arquivo oficial de 2025–2026 e o notebook principal não são alterados.

In [ ]:
from pathlib import Path
import json
import os
import time
import requests

def localizar_projeto():
    configurado = os.getenv('HELIO_PROJECT_ROOT')
    atual = Path(configurado).expanduser().resolve() if configurado else Path.cwd().resolve()
    for raiz in (atual, *atual.parents):
        if (raiz / 'manage.py').is_file():
            return raiz
    raise RuntimeError('Execute o notebook dentro do projeto Hélio.')

def carregar_env(caminho):
    if not caminho.is_file():
        return
    for linha in caminho.read_text(encoding='utf-8').splitlines():
        linha = linha.strip()
        if not linha or linha.startswith('#') or '=' not in linha:
            continue
        chave, valor = linha.split('=', 1)
        chave = chave.strip().lstrip(chr(65279))
        valor = valor.strip().strip(chr(34)).strip(chr(39))
        if not os.environ.get(chave):
            os.environ[chave] = valor

PROJECT_ROOT = localizar_projeto()
carregar_env(PROJECT_ROOT / '.env')
API_KEY = os.getenv('DATAJUD_API_KEY', '').strip()
if not API_KEY:
    raise RuntimeError('DATAJUD_API_KEY não encontrada no .env.')

ANO = 2000
LIMITE_POR_ANO = 1_000  # use None para coletar todo o ano 2000
URL_DATAJUD = 'https://api-publica.datajud.cnj.jus.br/api_publica_tjba/_search'
PASTA_RAW = PROJECT_ROOT / 'data' / 'raw' / 'datajud_teste' / str(ANO)
ARQUIVO_PARQUET = PROJECT_ROOT / 'data' / 'processed' / 'datajud' / 'teste' / 'tabela_principal_helios_2000_teste.parquet'
PASTA_RAW.mkdir(parents=True, exist_ok=True)
ARQUIVO_PARQUET.parent.mkdir(parents=True, exist_ok=True)

session = requests.Session()
session.headers.update({
    'Authorization': f'ApiKey {API_KEY}',
    'Content-Type': 'application/json',
    'Accept': 'application/json',
})

print(f'Projeto: {PROJECT_ROOT}')
print(f'Ano: {ANO}')
print(f'Limite de teste: {LIMITE_POR_ANO}')
print(f'Parquet de teste: {ARQUIVO_PARQUET}')

In [ ]:
SOURCE_FIELDS = [
    'grau', 'numeroProcesso', 'orgaoJulgador.codigo', 'orgaoJulgador.nome',
    'orgaoJulgador.codigoMunicipioIBGE', 'classe.codigo', 'classe.nome',
    'assuntos.codigo', 'assuntos.nome',
]
LIMITES = {'grau': 2, 'processo': 25, 'id_municipio': 7, 'municipio': 27, 'nome_classe': 310, 'codigo_orgao': 5, 'nome_orgao': 155}

def transformar(hit):
    source = hit.get('_source') or {}
    orgao = source.get('orgaoJulgador') or {}
    classe = source.get('classe') or {}
    assuntos = source.get('assuntos') or []
    if isinstance(assuntos, dict):
        assuntos = [assuntos]
    assuntos = [item for item in assuntos if isinstance(item, dict) and item.get('codigo') is not None]
    registro = {
        'ano': ANO, 'grau': str(source.get('grau') or ''),
        'processo': str(source.get('numeroProcesso') or ''),
        'id_municipio': str(orgao.get('codigoMunicipioIBGE') or ''),
        'municipio': str(source.get('municipio') or ''),
        'codigo_classe': str(classe.get('codigo') or ''),
        'nome_classe': str(classe.get('nome') or ''),
        'codigo_assunto': ' | '.join(str(item['codigo']) for item in assuntos),
        'nome_assunto': ' | '.join(str(item.get('nome') or '') for item in assuntos),
        'codigo_orgao': str(orgao.get('codigo') or ''),
        'nome_orgao': str(orgao.get('nome') or ''),
    }
    for campo, limite in LIMITES.items():
        if len(registro[campo]) > limite:
            raise ValueError(f'{campo} excede o limite do modelo: {len(registro[campo])} > {limite}')
    return registro

def buscar_pagina(search_after=None, lote=1_000):
    payload = {
        'size': lote, 'track_total_hits': False, '_source': SOURCE_FIELDS,
        'query': {'range': {'dataAjuizamento': {'gte': '20000101000000', 'lt': '20010101000000'}}},
        'sort': [{'id.keyword': {'order': 'asc'}}],
    }
    if search_after is not None:
        payload['search_after'] = search_after
    for tentativa in range(1, 8):
        try:
            resposta = session.post(URL_DATAJUD, json=payload, params={'allow_partial_search_results': 'false'}, timeout=(30, 300))
            if resposta.status_code == 200:
                dados = resposta.json()
                if dados.get('timed_out') or (dados.get('_shards') or {}).get('failed', 0):
                    raise RuntimeError('A API retornou timeout ou falha de shard.')
                hits = (dados.get('hits') or {}).get('hits')
                if not isinstance(hits, list):
                    raise RuntimeError('Resposta sem hits.hits.')
                return hits
            if resposta.status_code not in {429, 500, 502, 503, 504}:
                resposta.raise_for_status()
            raise RuntimeError(f'HTTP {resposta.status_code}')
        except (requests.RequestException, ValueError, RuntimeError) as erro:
            if tentativa == 7:
                raise
            espera = min(60, 2 ** tentativa)
            print(f'Tentativa {tentativa} falhou ({erro}); nova tentativa em {espera}s.')
            time.sleep(espera)


In [ ]:
ARQUIVO_PARCIAL = PASTA_RAW / 'tjba_2000_teste.jsonl.part'
ARQUIVO_FINAL = PASTA_RAW / 'tjba_2000_teste.jsonl'
CHECKPOINT = PASTA_RAW / 'checkpoint_2000_teste.json'

def salvar_checkpoint(estado):
    temporario = CHECKPOINT.with_suffix('.tmp')
    temporario.write_text(json.dumps(estado, ensure_ascii=False, indent=2), encoding='utf-8')
    temporario.replace(CHECKPOINT)

def coletar_teste():
    if ARQUIVO_FINAL.is_file() and ARQUIVO_FINAL.stat().st_size > 0:
        print(f'Coleta já existente: {ARQUIVO_FINAL}')
        return ARQUIVO_FINAL
    if CHECKPOINT.is_file():
        estado = json.loads(CHECKPOINT.read_text(encoding='utf-8'))
    elif ARQUIVO_PARCIAL.is_file() and ARQUIVO_PARCIAL.stat().st_size > 0:
        raise RuntimeError('Existe arquivo parcial sem checkpoint.')
    else:
        estado = {'search_after': None, 'processos': 0, 'file_size': 0, 'concluido': False}
    if ARQUIVO_PARCIAL.is_file() and ARQUIVO_PARCIAL.stat().st_size > estado['file_size']:
        with ARQUIVO_PARCIAL.open('r+b') as arquivo:
            arquivo.truncate(estado['file_size'])
    if estado['concluido'] and ARQUIVO_PARCIAL.is_file():
        ARQUIVO_PARCIAL.replace(ARQUIVO_FINAL)
        return ARQUIVO_FINAL
    total = int(estado['processos'])
    search_after = estado['search_after']
    if LIMITE_POR_ANO and total >= LIMITE_POR_ANO:
        return ARQUIVO_PARCIAL
    with ARQUIVO_PARCIAL.open('ab') as arquivo:
        while True:
            hits = buscar_pagina(search_after)
            if not hits:
                estado.update({'file_size': arquivo.tell(), 'concluido': True})
                salvar_checkpoint(estado)
                ARQUIVO_PARCIAL.replace(ARQUIVO_FINAL)
                print(f'Ano 2000 concluído: {total:,} registros')
                return ARQUIVO_FINAL
            novo_search_after = hits[-1].get('sort')
            if not novo_search_after or novo_search_after == search_after:
                raise RuntimeError('search_after não avançou.')
            for hit in hits:
                arquivo.write(json.dumps(transformar(hit), ensure_ascii=False, separators=(',', ':')).encode('utf-8') + b'\n')
            arquivo.flush()
            total += len(hits)
            search_after = novo_search_after
            estado.update({'search_after': search_after, 'processos': total, 'file_size': arquivo.tell(), 'concluido': False})
            salvar_checkpoint(estado)
            print(f'Registros coletados: {total:,}')
            if LIMITE_POR_ANO and total >= LIMITE_POR_ANO:
                print('Limite de teste atingido; o arquivo parcial foi preservado.')
                return ARQUIVO_PARCIAL

coletar_teste()

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

COLUNAS = ['ano', 'grau', 'processo', 'id_municipio', 'municipio', 'codigo_classe', 'nome_classe', 'codigo_assunto', 'nome_assunto', 'codigo_orgao', 'nome_orgao']
SCHEMA = pa.schema([pa.field('ano', pa.int16()), *[pa.field(campo, pa.string()) for campo in COLUNAS[1:]]])

def consolidar_teste():
    if ARQUIVO_FINAL.is_file():
        caminho = ARQUIVO_FINAL
        limite = caminho.stat().st_size
    else:
        caminho = ARQUIVO_PARCIAL
        limite = json.loads(CHECKPOINT.read_text(encoding='utf-8'))['file_size']
    temporario = Path(str(ARQUIVO_PARQUET) + '.tmp')
    lote = []
    total = 0
    with pq.ParquetWriter(temporario, SCHEMA, compression='snappy') as escritor:
        with caminho.open('rb') as arquivo:
            while arquivo.tell() < limite:
                linha = arquivo.readline(limite - arquivo.tell())
                if not linha:
                    break
                lote.append(json.loads(linha.decode('utf-8')))
                total += 1
                if len(lote) >= 1_000:
                    escritor.write_table(pa.Table.from_pylist(lote, schema=SCHEMA))
                    lote.clear()
        if lote:
            escritor.write_table(pa.Table.from_pylist(lote, schema=SCHEMA))
    validacao = pq.ParquetFile(temporario)
    linhas_validadas = validacao.metadata.num_rows
    validacao.close()
    if linhas_validadas != total:
        temporario.unlink(missing_ok=True)
        raise RuntimeError('Quantidade de linhas inválida no Parquet de teste.')
    temporario.replace(ARQUIVO_PARQUET)
    print(f'Parquet de teste salvo: {ARQUIVO_PARQUET}')
    print(f'Registros: {total:,}')
    return ARQUIVO_PARQUET

consolidar_teste()

In [ ]:
parquet = pq.ParquetFile(ARQUIVO_PARQUET)
print(f'Arquivo: {ARQUIVO_PARQUET}')
print(f'Linhas: {parquet.metadata.num_rows:,}')
print('Colunas:', parquet.schema_arrow.names)
print('Amostra:', parquet.read_row_group(0).slice(0, 5).to_pylist())
print('Teste concluído. Para validar o importador sem gravar no banco:')
print(f'python manage.py importar_datajud --arquivo {ARQUIVO_PARQUET} --dry-run')